# Imbalanced Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadtalhaishtiaq/ai-orchestrator/blob/main/01-data-preprocessing/06_imbalanced_data.ipynb)


---

## What are we learning?

We’ll see what happens when one class in your dataset far outnumbers the other, why accuracy can fool you, and how to fix the imbalance with simple resampling tricks.

## The idea in plain English

Imagine a cookie jar with 950 chocolate-chip cookies and only 50 oatmeal-raisin ones. If you grab a handful and always guess “chocolate-chip”, you’ll be right 95 % of the time—but you’ll never notice the oatmeal-raisin lovers! Imbalanced data is the same: the model gets rewarded for ignoring the minority class. Today we’ll even things out so every cookie (class) gets a fair chance.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

sns.set_theme(style="whitegrid")
print('Setup done!')

## Step 1 — Load data

In [ ]:
X, y = make_classification(n_samples=1000, n_features=20, n_informative=2,
                           n_redundant=10, n_clusters_per_class=1,
                           weights=[0.95], flip_y=0, random_state=42)

df = pd.DataFrame(X, columns=[f'feat_{i}' for i in range(X.shape[1])])
df['target'] = y
print(df.shape)
df['target'].value_counts()

## Step 2 — Apply Imbalanced Data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,
                                                    random_state=42, stratify=y)

clf_raw = LogisticRegression(max_iter=1000, random_state=42)
clf_raw.fit(X_train, y_train)

y_pred = clf_raw.predict(X_test)
print("Raw (imbalanced) dataset:")
print(classification_report(y_test, y_pred))

## Step 3 — Balance the training set with oversampling

In [ ]:
ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

print("After oversampling:")
print(pd.Series(y_train_bal).value_counts().sort_index())

## Step 4 — Train again on balanced data

In [ ]:
clf_bal = LogisticRegression(max_iter=1000, random_state=42)
clf_bal.fit(X_train_bal, y_train_bal)

y_pred_bal = clf_bal.predict(X_test)
print("Balanced dataset:")
print(classification_report(y_test, y_pred_bal))

## Step 3 — Visualise

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Original distribution
sns.countplot(x=pd.Series(y_train), ax=ax[0], palette="Set2")
ax[0].set_title("Original training set")
ax[0].set_xlabel("Class")

# Balanced distribution
sns.countplot(x=pd.Series(y_train_bal), ax=ax[1], palette="Set2")
ax[1].set_title("Oversampled training set")
ax[1].set_xlabel("Class")

plt.tight_layout()
plt.show()

## Results & interpretation

In [ ]:
print("Confusion matrix (imbalanced training):")
print(confusion_matrix(y_test, y_pred))
print("\nConfusion matrix (balanced training):")
print(confusion_matrix(y_test, y_pred_bal))

# Notice how the balanced model catches far more of the minority class (label 1).

## Summary

- Imbalanced datasets make models lazy—high accuracy hides poor minority-class recall.
- Random oversampling duplicates minority examples to even class counts.
- After balancing, recall for the minority class jumps (check the confusion matrix).
- Always inspect precision & recall, not just accuracy, when classes are skewed.
- imblearn gives one-line resampling tools like RandomOverSampler.

## Exercises

1. Try RandomUnderSampler instead of oversampling and compare recall scores.
2. Use SMOTE (from imblearn.over_sampling) for synthetic minority samples and plot the new 2-D PCA space.
3. Combine over- and under-sampling (e.g., EditedNearestNeighbours + RandomOverSampler) and report F1-score.

In [ ]:
# Your code here